In [1]:
# Install modules
!pip install gspread==3.7.0 -q
!pip install gspread-dataframe -q

# import mods
import gspread
import pandas as pd
from google.colab import auth
from google.auth import default
from gspread_dataframe import get_as_dataframe, set_with_dataframe

In [2]:
class process:

  ############################################################################ Set Init  ##################################################################
  def __init__(self):

    # Auth user
    auth.authenticate_user()

    # Cred to default
    creds, _ = default()

    # objected gc
    self.gc = gspread.authorize(creds)

    # Print Connection
    print('Conexão autenticada | BrainBot')
    print(f'gspread version: {gspread.__version__}')

    # import mapping reference
    df_url = 'https://docs.google.com/spreadsheets/d/1HyGSR03GDhKlc4bvcs1GDvLpsi5RWrt5wo5J9-99xCs/edit?usp=sharing'
    self.sh = self.gc.open_by_url(df_url)
    self.ws = self.sh.worksheet("Remapeamento")
    self.remap = pd.DataFrame(self.ws.get_all_records()).set_index('Codif')

    # import database
    # Input de url
    self.sh = self.gc.open_by_url(df_url)
    self.ws = self.sh.worksheet("Base")
    self.df = pd.DataFrame(self.ws.get_all_records()).set_index('Código').fillna('-')


    # Tratamentos

    # Geração
    def categorize_generation(age):
      if age == '-':
          return '-'
      elif age <= 17:
          return 'Geração Alfa'
      elif age >= 18 and age <= 27:
          return 'Geração Z'
      elif age >= 28 and age <= 43:
          return 'Geração Y'
      elif age >= 44 and age <= 59:
          return 'Geração X'
      elif age >= 60 and age <= 78:
          return 'Boomers'
      else:
          return 'Geração Silenciosa'

    # Aplicar a função para criar a nova coluna 'Geração'
    self.df['Geracao'] = self.df['FE2P5'].apply(categorize_generation)

    # Renda
    def categorize_income_range(income):
      if income == '-':
          return '-'
      elif income in ['Até R$ 1.500,00', 'De R$ 1.500,00 a R$ 2.500,00', 'De R$ 2.500,01 a R$ 4.500,00', 'De R$ 4.500,01 a R$ 5.500,00']:
          return 'De R$ 1.500,00 a R$ 5.500,00'
      elif income in ['De R$ 5.500,01 a R$ 8.000,00', 'De R$ 8.000,01 a R$ 11.000,00']:
          return 'De R$ 5.500,01 a R$ 11.000,00'
      elif income in ['De R$ 11.000,01 a R$ 13.000,00', 'De R$ 13.000,01 a R$ 16.000,00']:
          return 'De R$ 11.000,01 a R$ 16.000,00'
      elif income in ['De R$ 16.000,01 a R$ 18.500,00', 'De R$ 18.500,01 a R$ 21.000,00']:
          return 'De R$ 16.000,01 a R$ 21.000,00'
      else:
          return 'Acima de R$ 21.000,00'

    # Aplicar a função para criar a nova coluna 'Faixa de Renda'
    self.df['Renda'] = self.df['FE2P10'].apply(categorize_income_range)

    # Regiao
    def categorize_region(state):
      if state == '-':
          return '-'
      elif state in ['AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO']:
          return 'Norte'
      elif state in ['AL', 'BA', "Bahia", 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE']:
          return 'Nordeste'
      elif state in ['DF', 'GO', 'MT', 'MS']:
          return 'Centro-Oeste'
      elif state in ['ES', 'MG', 'RJ', 'SP']:
          return 'Sudeste'
      elif state in ['PR', 'RS', 'SC']:
          return 'Sul'
      else:
          return 'Desconhecido'

    # Aplicar a função para criar a nova coluna 'Região'
    self.df['Regiao'] = self.df['Estado'].apply(categorize_region)

    # Tipo de Estudo
    def categorize_study_type(value):
      return 'Horizontal' if value == '-' else 'Vertical'

    # Aplicar a função para criar a nova coluna 'TipoEstudo'
    self.df['TipoEstudo'] = self.df['IIA5P43'].apply(categorize_study_type)

    # Categorias gerais para as areas
    def categorize_common_area(value):
      if value in ['Academia', 'Academia ao ar livre', 'Quadra Poliesportiva', 'Quadra de Beach Tenis', 'Pista de caminhada', 'Ciclovia']:
          return 'Esporte & Atividades Físicas'
      elif value in ['Salão de Festas', 'Espaço Gourmet', 'Churrasqueira', 'Churrasqueira com deck/bar', 'Praça de eventos (food truck/ feira orgânicos/ festa junina, ect)', 'Rooftop', 'Sala de jogos']:
          return 'Áreas de Convivência & Festas'
      elif value in ['Playground', 'Parquinho', 'Praça']:
          return 'Áreas Infantis & Familiares'
      elif value in ['Piscina', 'Piscina Adulto e Infantil', 'Piscina adulto', 'Piscina e Deck', 'Spa', 'Sauna']:
          return 'Áreas Aquáticas & Relaxamento'
      elif value in ['Portaria', 'Espaço pet', 'Pet Place', 'Longe do Centro', 'Longe do meu local de trabalho', 'Tem mais interesse em outra região', 'Alphaville']:
          return 'Infraestrutura & Segurança'
      elif value in ['-']:
          return '-'
      else:
          return 'Outros'

    # Aplicar a função para as colunas APAC9P85_1 a APAC9P85_5
    for col in ['APAC9P85_1', 'APAC9P85_2', 'APAC9P85_3', 'APAC9P85_4', 'APAC9P85_5']:
        self.df[col + '_Categoria'] = self.df[col].apply(categorize_common_area)

        # Categorias 2 gerais para as areas
    def categorize_common_area2(value):

      # E&AF
      if value in ['Academia', 'Academia ao ar livre', ]:
          return 'E&AF | Academias'

      elif value in ['Quadra Poliesportiva', 'Quadra de Beach Tenis']:
          return 'E&AF | Quadras'

      elif value in ['Pista de caminhada', 'Ciclovia']:
          return 'E&AF | Pistas Caminhada e Ciclovia'

      # AC&F
      elif value in ['Salão de Festas', 'Espaço Gourmet', 'Sala de jogos']:
          return 'AC&F | Ambientes fechados'

      elif value in ['Praça de eventos (food truck/ feira orgânicos/ festa junina, ect)', 'Rooftop']:
          return 'AC&F | Ambientes abertos'

      elif value in ['Churrasqueira', 'Churrasqueira com deck/bar']:
          return 'AC&F | Churrasqueiras'

      # AI&F
      elif value in ['Playground', 'Parquinho', 'Praça']:
          return 'Áreas Infantis & Familiares'

      # AA&R
      elif value in ['Piscina', 'Piscina Adulto e Infantil', 'Piscina adulto', 'Piscina e Deck']:
          return 'AA&R | Piscinas'

      elif value in [ 'Spa', 'Sauna']:
          return 'AA&R | Sauna e SPA'

      # IP
      elif value in ['Portaria', 'Espaço pet', 'Pet Place', 'Longe do Centro', 'Longe do meu local de trabalho', 'Tem mais interesse em outra região', 'Alphaville']:
          return 'Infraestrutura Pet'

      elif value in ['-']:
          return '-'

      else:
          return 'Outros'

    # Aplicar a função para as colunas APAC9P85_1 a APAC9P85_5
    for col in ['APAC9P85_1', 'APAC9P85_2', 'APAC9P85_3', 'APAC9P85_4', 'APAC9P85_5']:
        self.df[col + '_Cat2'] = self.df[col].apply(categorize_common_area2)


    # SET BLOCOS
    self.perfil = ['FE2P3','FE2P5', 'FE2P6', 'Geracao', 'Cidade', 'Estado', 'Regiao',
                   'FE2P10', 'Renda', 'PS3P17','PS3P19', 'PS3P23', 'PS3P26']

    self.intencao = ['IC4P30', 'IC4P32',
                     ['IC4P33_1','IC4P33_2'],
                     ['IC4P34_1','IC4P34_2'],
                     'IC4P35']

    self.ideal = ['IIA5P37', 'IIA5P39', 'IIA5P41', 'IIA5P47',
                  'IIA5P43', 'IIA5P44', 'IIA5P45', 'IIA5P46'
                  ]

    self.areas = [['APAC9P85_1', 'APAC9P85_2', 'APAC9P85_3',
                  'APAC9P85_4', 'APAC9P85_5'], 'APAC9P86', 'TipoEstudo'
                  ]

    self.areasReduz = [['APAC9P85_1_Categoria', 'APAC9P85_2_Categoria', 'APAC9P85_3_Categoria',
                  'APAC9P85_4_Categoria', 'APAC9P85_5_Categoria']
                  ]

    self.areasReduz2 = [['APAC9P85_1_Cat2', 'APAC9P85_2_Cat2', 'APAC9P85_3_Cat2',
                  'APAC9P85_4_Cat2', 'APAC9P85_5_Cat2']
                  ]

    self.compra12m = ['CNM14P128',	['CNM14P129_1',	'CNM14P129_2'],
                     'CNM14P130',	'CNM14P131',	'CNM14P134']

    self.imovelatual = ['IA15P135',	'IA15P136',	'IA15P138',
                      'IA15P139',	'IA15P140',	'IA15P141']


    # SET
    self.recorte_dfs = [self.df[self.df.TipoEstudo == 'Vertical'], self.df[self.df.TipoEstudo == 'Horizontal']]

    # SET BLOCOS DICT
    self.blocos_dict = {#'Perfil': self.perfil,
                        #'Intenção de Compra': self.intencao,
                        #'Imóvel de Interesse': self.ideal,
                        'Áreas Comuns Lazer': self.areas,
                        'Áreas Reduzidas': self.areasReduz,
                        'Áreas Reduzidas 2': self.areasReduz2,
                        #'Comprou Imóvel | 12 meses': self.compra12m,
                        #'Imóvel atual': self.imovelatual,
                        }

  ###################################################################################################################################################################
  ###################################################################################################################################################################
  ###################################################################################################################################################################

  ############################################################################ Set function decode ##################################################################
  def decode(self, cod):
    '''
    Função para decodificar as perguntas da base do processamento
    decode(cod) retorna a pergunta original para o código 'cod'
    '''

    # Set
    map = self.remap.loc[cod].values[0]
    #orig = self.remap.loc[cod].values[1]

    # Print
    return (map,'orig')


  ############################################################################ Set function create process ##################################################################
  def criar_processamento(self):
    '''
    Função para criar a planilha de processamento automatizada.
    Inputs: ID do folder de Apresentação (link)
    '''

    # Input de ID
    self.apres_id = '1xvfnS_9NqhKZ2xesrz20lBHkFuj3JoTq'

    # Create file name
    self.process_name = 'Processamento Base Unificada 2024 | Areas P2'

    # Check if exists
    try:
      self.gc.open(self.process_name)
      print(f'A planilha já existe | {self.process_name}')

    except:
      # Create file
      self.gc.create(self.process_name, folder_id=self.apres_id)


  ############################################################################ Set function access process ##################################################################
  def acessar_processamento(self):
    '''
    Função para acessar a planilha de processamento automatizada.
    '''

    # Create object
    try:
      self.planilha = self.gc.open(self.process_name)
      print('Planilha acessada')

    except:
      print('A planilha já está aberta ou não foi criada.')


  # Counter handling functions
  #######################################################################################################################################################################

  ################## define next row ##########################################
  def next_available_row(self, worksheet):
      str_list = list(worksheet.col_values(2))
      return len(str_list) + 3


  ################## define contador(col) ################################
  def contador(self, df, att, add=False, ws='undef.'):

    # Average
    if att == 'FE2P5':

      # Add a new df with the average age
      temp_df = pd.DataFrame({'Name': ['Média'], 'Idade': round(df['FE2P5'].mean(), 0)})

    else:
      # Set calculations dataframe /  Drop '-' non-respondents
      temp_df = pd.DataFrame(df[att].value_counts())

      try: temp_df.drop('-', inplace = True)
      except: pass

      # Set counting reference
      tot_sum = temp_df['count'].sum()

      # Get percentage
      temp_df['Percentage'] = round(temp_df['count'] / tot_sum, 2)

      # Adjust sorting
      temp_df.index = temp_df.index.map(str)
      temp_df = temp_df[['Percentage', 'count']].sort_index(ascending=True)

    # Decode
    temp_df.rename(columns={'count':self.decode(att)[0]}, inplace=True)

    # Add to sheet
    if add:
      set_with_dataframe(ws, temp_df, row=self.next_available_row(ws), col=1,
                        include_index=True, include_column_header=True)
    else: print(temp_df,'\n\n ##### ##### ##### ##### ##### ##### ##### #####')


  ################## define multiple value_counter(col) #######################
  def contadorM(self, df, mlst, add=False, ws='undef.'):

    # Check if mlst has more than one column
    if len(mlst) == 0:
        print("Error: 'mlst' must contain at least one column.")
        return

    # Helper function to process individual columns
    def process_column(col):
        temp_df = pd.DataFrame(df[col].value_counts())
        temp_df = temp_df[temp_df.index != '-']  # Removing '-' in a more concise way
        return temp_df

    # Initialize with the first column
    comp_df = process_column(mlst[0])
    ref_div = comp_df.sum().iloc[0]

    # Iterate through the remaining columns in mlst
    for att in mlst[1:]:
        temp_df = process_column(att)
        try:
            comp_df = comp_df.join(temp_df, rsuffix='_temp')
        except KeyError as e:
            print(f"Join failed for column {att}: {e}")

    # Fill NaN values and sum rows
    comp_df.fillna(0, inplace=True)
    comp_df['Total Sum'] = comp_df.sum(axis=1)

    # Calculate percentages
    comp_df['Percentage Total'] = round(comp_df['Total Sum'] / ref_div, 2)
    comp_df['Percentage First'] = round(comp_df[comp_df.columns[0]] / ref_div, 2)

    # Sort by 'Percentage Total'
    comp_df = comp_df.sort_values('Percentage Total', ascending=False)

    # Reorganize columns to desired order
    column_order = ['Percentage Total', 'Total Sum', 'Percentage First'] + \
                   [col for col in comp_df.columns if 'count' in col]
    comp_df = comp_df[column_order]


    # Output or add to worksheet
    if add:
        set_with_dataframe(ws, comp_df, row=self.next_available_row(ws), col=1,
                           include_index=True, include_column_header=True)
    else:
        print(comp_df, '\n\n ##### ##### ##### ##### ##### ##### ##### #####')


  ############################## define cutter ################################
  def cutter(self, df, recorte, objeto, add=False,  ws='undef.'):

    #Skip
    if objeto in ['FE2P5', 'FE2P7', 'IC4P35', 'IC4P37', 'FE2P7',
                  'CNM14P131', 'CNM14P132', 'CNM14P133']: return

    # Agrupa por recorte e objeto
    grouped = df.groupby([recorte, objeto]).size().reset_index(name='Count')

    # Seta pivot
    pivot_table = pd.pivot_table(grouped, index=recorte, columns=objeto, values='Count').fillna(0)

    # Calculating the sum for each column and creating the "Geral" row
    geral = pivot_table.sum()

    # Adding the "Geral" row to the DataFrame
    pivot_table.loc['Geral'] = geral

    try: pivot_table.drop('-',  axis=1, inplace = True)
    except: pass

    # %
    rows_lst = []
    for row in range(len(pivot_table)):
      rows_lst.append(pivot_table.iloc[row].values)

    i=0
    for col in pivot_table:
      pivot_table[f'%_{col}'] = [
                                  round(row_values[i]/row_values.sum(), 2) if row_values.sum() != 0 else 0
                                  for row_values in rows_lst
                                ]
      i+=1

    # Filtrar colunas que contêm "%" no nome
    pivot_table = pivot_table.filter(like='%')

    # Renomear colunas removendo o sufixo "_%"
    pivot_table = pivot_table.rename(columns=lambda x: x.replace('%_', ''))
    pivot_table.index.name = self.decode(objeto)[0]
    pivot_table = pivot_table.sort_index(ascending=False)

    if add:
      set_with_dataframe(ws, pivot_table, row=self.next_available_row(ws), col=1, include_index=True, include_column_header=True)

    if add == False: return pivot_table

  ###################################################################################################################################################################
  ################################################################## FUNÇÕES ANALITICAS #############################################################################
  ###################################################################################################################################################################

  ############################################################################ Set PERFIL ##################################################################
  def preencher_bloco(self, bloco, add=False, overwrite=False):
    f'''
    Função para criar, acessar e preencher a página do bloco {bloco} do questionário.
    '''

    # Create worksheet
    try:
      self.planilha.add_worksheet(bloco, rows=350, cols=20)
      print(f'Página {bloco} adicionada.')

    except:
      print(f'A página {bloco} já existe neste processamento.')

    if overwrite:
      add = True
      self.planilha.worksheet(bloco).clear()

    # Fill through Perfil list loop
    for each in self.blocos_dict.get(bloco):

      if type(each) == list:
        self.contadorM(self.df, each, add=add, ws=self.planilha.worksheet(bloco))

      else:
        try: self.contador(self.df, each, add=add, ws=self.planilha.worksheet(bloco))
        except: pass

    print(f'Página {bloco} Preenchida.')


  ############################################################################ Set PERFIL RECORTES ##################################################################
  def preencher_recortes(self, bloco, add=False, overwrite=False):
    f'''
    Função para criar, acessar e preencher a página do bloco {bloco} | Recortes do questionário.
    '''

    # Create worksheet
    try:
      self.planilha.add_worksheet(f'{bloco} | Recortes', rows=350, cols=20)
      print(f'Página {bloco} | Recortes adicionada.')

    except:
      print(f'A página {bloco} | Recortes já existe neste processamento.')

    if overwrite:
      add = True
      self.planilha.worksheet(f'{bloco} | Recortes').clear()

    # Fill through Perfil | Recortes list loop
    for df in self.recorte_dfs:

      for each in self.blocos_dict.get(bloco):

        if type(each) == list:
          self.contadorM(df, each, add=add, ws=self.planilha.worksheet(f'{bloco} | Recortes'))

        else:
          try: self.contador(df, each, add=add, ws=self.planilha.worksheet(f'{bloco} | Recortes'))
          except: pass

    print(f'Página {bloco} | Recortes Preenchida.')


  ############################################################################ Set PERFIL CUTS ##################################################################
  def preencher_cuts(self, bloco, add=False, overwrite=False):
    f'''
    Função para criar, acessar e preencher a página do bloco {bloco} | Cuts do questionário.
    '''

    # Create worksheet
    try:
      self.planilha.add_worksheet(f'{bloco} | Cuts', rows=350, cols=20)
      print(f'Página {bloco} | Cuts adicionada.')

    except:
      print(f'A página {bloco} | Cuts já existe neste processamento.')

    if overwrite:
      add = True
      self.planilha.worksheet(f'{bloco} | Cuts').clear()

    # Fill through Perfil | Cuts list loop
    cut_list = ['Regiao', 'Renda', 'Geracao', 'TipoEstudo']

    for cut in cut_list:
      for each in self.blocos_dict.get(bloco):
        if type(each) == list:
          try: self.cutter(self.df, cut, each[0], add=add, ws=self.planilha.worksheet(f'{bloco} | Cuts'))
          except: pass

        else:
          try: self.cutter(self.df, cut, each, add=add, ws=self.planilha.worksheet(f'{bloco} | Cuts'))
          except: pass

    print(f'Página {bloco} | Cuts Preenchida.')



### MAIN ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ###
process = process()

Conexão autenticada | BrainBot
gspread version: 3.7.0


In [ ]:
process.criar_processamento()
process.acessar_processamento()

for bloco in process.blocos_dict:
  process.preencher_bloco(bloco, overwrite=True)
  process.preencher_recortes(bloco, overwrite=True)
  process.preencher_cuts(bloco, overwrite=True)

A planilha já existe | Processamento Base Unificada 2024 | Areas P2
Planilha acessada
Página Áreas Comuns Lazer adicionada.
Página Áreas Comuns Lazer Preenchida.
Página Áreas Comuns Lazer | Recortes adicionada.
Página Áreas Comuns Lazer | Recortes Preenchida.
A página Áreas Comuns Lazer | Cuts já existe neste processamento.
Página Áreas Comuns Lazer | Cuts Preenchida.
Página Áreas Reduzidas adicionada.
Página Áreas Reduzidas Preenchida.
A página Áreas Reduzidas | Recortes já existe neste processamento.
Página Áreas Reduzidas | Recortes Preenchida.
A página Áreas Reduzidas | Cuts já existe neste processamento.
Página Áreas Reduzidas | Cuts Preenchida.
A página Áreas Reduzidas 2 já existe neste processamento.
Página Áreas Reduzidas 2 Preenchida.
A página Áreas Reduzidas 2 | Recortes já existe neste processamento.
Página Áreas Reduzidas 2 | Recortes Preenchida.
A página Áreas Reduzidas 2 | Cuts já existe neste processamento.
Página Áreas Reduzidas 2 | Cuts Preenchida.


# explore

In [21]:
import plotly.express as px
explore_df = process.df

In [46]:
# Filtrar dados apenas para certeza máxima de compra
df_filtered = process.df[
    (process.df["Geracao"] == "Geração Z") &
    (process.df["APAC9P85_1_Categoria"] != "-") &
    (process.df["APAC9P86"] != "-")# Remover valores inválidos
]

# Criar a tabela cruzada (pivot)
cross_table = pd.crosstab(
    df_filtered["APAC9P85_1_Categoria"],
    df_filtered["APAC9P86"],
    normalize="index"  # Calcula porcentagens dentro de cada tipo de empreendimento
)  # Converter para porcentagem

# Ordenar as colunas por maior impacto no estudo

cross_table

APAC9P86,1. Certamente não compraria,2. Provavelmente não compraria,3. Não tem certeza se compraria,4. Provavelmente compraria,5. Certamente compraria
APAC9P85_1_Categoria,,,,,
Esporte & Atividades Físicas,0.025765,0.006441,0.045089,0.352657,0.570048
Infraestrutura & Segurança,0.031373,0.015686,0.047059,0.282353,0.623529
Áreas Aquáticas & Relaxamento,0.012346,0.014403,0.028807,0.308642,0.635802
Áreas Infantis & Familiares,0.024272,0.019417,0.033981,0.349515,0.572816
Áreas de Convivência & Festas,0.014184,0.011820,0.028369,0.307329,0.638298


In [48]:
# Filtrar dados apenas para certeza máxima de compra
df_filtered = process.df[
    (process.df["Geracao"] == "Boomers") &
    (process.df["APAC9P85_1_Categoria"] != "-") &
    (process.df["APAC9P86"] != "-")# Remover valores inválidos
]

# Criar a tabela cruzada (pivot)
cross_table = pd.crosstab(
    df_filtered["APAC9P85_1_Categoria"],
    df_filtered["APAC9P86"],
    normalize="index"  # Calcula porcentagens dentro de cada tipo de empreendimento
)  # Converter para porcentagem

# Ordenar as colunas por maior impacto no estudo

cross_table

APAC9P86,1. Certamente não compraria,2. Provavelmente não compraria,3. Não tem certeza se compraria,4. Provavelmente compraria,5. Certamente compraria
APAC9P85_1_Categoria,,,,,
Esporte & Atividades Físicas,0.034146,0.014634,0.087805,0.395122,0.468293
Infraestrutura & Segurança,0.085271,0.031008,0.108527,0.418605,0.356589
Áreas Aquáticas & Relaxamento,0.008475,0.016949,0.046610,0.300847,0.627119
Áreas Infantis & Familiares,0.140351,0.000000,0.052632,0.421053,0.385965
Áreas de Convivência & Festas,0.052632,0.012146,0.052632,0.348178,0.534413


In [45]:
explore_df.Geracao

,Geracao
Código,
4670538,Geração Z
4668070,Geração X
4667266,Geração X
4670539,Geração Y
4670540,Geração Y
...,...
6845957,-
6877824,-
6878571,-
